In [ ]:
### CONSTANTS AND HELPERS
import sqlite3, io, pandas as pd, base64, matplotlib.pyplot as plt, sys, os, glob, pytz, IPython.core.display as ip, plotly.express as px
from IPython.display import display, HTML
from datetime import datetime

surveyYear = '2025'
aboveDamOnly = False
use_v2_report = True
use_compiled_db = True

additionalFilterForAboveDam = "AND CAST(Distance AS int) > 310" if aboveDamOnly else ""

surveyURIs = {'2019':'https://five.epicollect.net/api/export/entries/salmon-survey-2019?form_ref=397fba6ecc674b74836efc190840c42d_5d6f454667a28&per_page=100',
              '2020':'https://five.epicollect.net/api/export/entries/salmon-survey-2020?form_ref=f550ab6c4dab44f49bcc33b7c1904be9_5d6f454667a28&per_page=100',
              '2021':'https://five.epicollect.net/api/export/entries/salmon-survey-2021?form_ref=ad5ffedf0a3246a18934e6ec36ed9569_5d6f454667a28&per_page=100',
              '2022':'https://five.epicollect.net/api/export/entries/salmon-survey-2022?form_ref=d46b5d8451f8410ea407bae5c8eb9f49_5d6f454667a28&per_page=100'}
salmonURIs = {'2019':'https://five.epicollect.net/api/export/entries/salmon-survey-2019?form_ref=397fba6ecc674b74836efc190840c42d_5d6f509867795&per_page=500',
              '2020':'https://five.epicollect.net/api/export/entries/salmon-survey-2020?form_ref=f550ab6c4dab44f49bcc33b7c1904be9_5d6f509867795&per_page=500',
              '2021':'https://five.epicollect.net/api/export/entries/salmon-survey-2021?form_ref=ad5ffedf0a3246a18934e6ec36ed9569_5d6f509867795&per_page=500',
              '2022':'https://five.epicollect.net/api/export/entries/salmon-survey-2022?form_ref=d46b5d8451f8410ea407bae5c8eb9f49_5d6f509867795&per_page=500',
              '2023':'https://kf.kobotoolbox.org/api/v2/assets/a6dEG7tnrtwjrmituAdL5k/data/?format=json',
              '2024':'https://kf.kobotoolbox.org/api/v2/assets/ae8BCoHi4EmwnzP2ShmSUw/data/?format=json',
              '2025':'https://kf.kobotoolbox.org/api/v2/assets/a5WFFCGawCP3aTLHdRjrca/data/?format=json'
             }

plotly_font_family = {'family': "Arial, Helvetica, sans-serif"}
plotly_year_menu_attrs = {
    'type': 'buttons',
    'direction': 'right',
    'showactive': True,
    'xanchor': 'left',
    'yanchor': 'top',
    'bgcolor': 'lightgrey',
    'bordercolor': 'grey',
    'borderwidth': 1
}

IN_COLAB = 'google.colab' in sys.modules



In [ ]:
### DB AND TABLE SETUP
from data_helper import DataHelper
from report_helper import ReportHelper

import importlib
import report_helper
import data_helper
importlib.reload(report_helper)
importlib.reload(data_helper)
from report_helper import ReportHelper
from data_helper import DataHelper

#for running locally
def clearPreviousReports():
    for fileName in glob.glob('*salmonReport.html'):
        print(f"Previous report file exists. Deleting {fileName}")
        os.remove(fileName)

def getAllFiles():
    !git clone https://github.com/slfisco/Survey-Notebook.git

if IN_COLAB:
    getAllFiles()
else:
    clearPreviousReports()
dataHelper = reportHelper = reddsTable = yearByYearCountPlot = countPlot = surveyStatsTable = yearScatterMap = latestScatterMap = None
maxSurveyChum = maxSurveyChumDate = maxSurveyCoho = maxSurveyCohoDate = chumSpawnSuccess = cohoSpawnSuccess = chumFemaleSpawnSuccess = chumMaleSpawnSuccess = chumPredation = cohoPredation = None   
dataHelper = DataHelper(surveyYear=surveyYear, allYears=list(salmonURIs.keys()), aboveDamOnly=aboveDamOnly, inCollab=IN_COLAB)
reportHelper = ReportHelper(dataHelper=dataHelper)

In [ ]:
### DATA LOADING
print('loading salmon into database')
dataHelper.getDataV1()
# dataHelper.getDataV2()
connection = dataHelper.connection

In [ ]:
df = dataHelper.getSurveyStatsV1(surveyYear)
maxSurveyChum = dataHelper.getMaxSurveyTotal(df, 'running_total_all_chum')
maxSurveyChumDate = dataHelper.getMaxSurveyDate(df, 'running_total_all_chum')
maxSurveyCoho = dataHelper.getMaxSurveyTotal(df, 'running_total_all_coho')
maxSurveyCohoDate = dataHelper.getMaxSurveyDate(df, 'running_total_all_coho')
print(f'max survey chum: {maxSurveyChum}')
print(f'max survey chum date: {maxSurveyChumDate}')
print(f'max survey coho: {maxSurveyCoho}')
print(f'max survey coho date: {maxSurveyCohoDate}')

In [ ]:
surveyStatsTable = reportHelper.displaySurveyStatsTable()
display(ip.HTML(surveyStatsTable))

In [ ]:
### DAILY FISH COUNT BY TYPE AND SPECIES
if use_v2_report:
    countPlotByYear = reportHelper.displayCountPlotChartByYear()
    display(HTML(countPlotByYear))
else:
    countPlot = reportHelper.displayCountPlot(dataHelper.getSurveyStatsV1(dataHelper.surveyYear))
    display(HTML(countPlot))


In [ ]:
### REDDS TABLE. USED TO HELP SURVEY TEAM AVOID REDDS
reddsTable = reportHelper.plotReddsTable()
display(ip.HTML(reddsTable))

In [ ]:
### SPAWN SUCCESS
if use_v2_report:
    spawningChartsByYear = reportHelper.displaySpawningChartsByYear()
    display(HTML(spawningChartsByYear))
else:
    chumSpawnSuccess = reportHelper.plotSpawning('Chum')
    chumMaleSpawnSuccess = reportHelper.plotSpawning('Chum', 'Male')
    chumFemaleSpawnSuccess = reportHelper.plotSpawning('Chum', 'Female')
    cohoSpawnSuccess = reportHelper.plotSpawning('Coho')
    display(HTML(chumSpawnSuccess))
    display(HTML(chumMaleSpawnSuccess))
    display(HTML(chumFemaleSpawnSuccess))
    display(HTML(cohoSpawnSuccess))



In [ ]:
### PREDATION
if use_v2_report:
    predationChartsByYear = reportHelper.displayPredationChartsByYear()
    display(HTML(predationChartsByYear))
else:
    chumPredation = reportHelper.plotPredation('Chum')
    cohoPredation = reportHelper.plotPredation('Coho')

In [ ]:
# ## USER INPUT QUERY
try:
    # query = '''
    #     select
    #         COALESCE(SUM(CASE WHEN Type = 'Live' THEN Quantity END), 0) AS live_count,
    #         COALESCE(SUM(CASE WHEN Type = 'Dead' THEN Quantity END), 0) AS dead_count,
    #         COALESCE(SUM(CASE WHEN Type = 'Remnant' THEN Quantity END), 0) AS remnant_count,
    #         COALESCE(SUM(CASE WHEN Type = 'Redd' THEN Quantity END), 0) AS redd_count,
    #         COALESCE(SUM(CASE WHEN Species = 'Chum' THEN Quantity END), 0) AS chum_count,
    #         COALESCE(SUM(CASE WHEN Species = 'Coho' THEN Quantity END), 0) AS coho_count
    #     FROM salmon
    #     where Year=2025;
    # '''
    query = 'select distinct Year from survey_data order by Year desc;'
    # print("entering query: " + query)
    dataHelper.connection.execute(query)
    print(dataHelper.connection.execute(query).fetchall())
except sqlite3.Error as e:
    print("SQLite error:", e)

In [ ]:
import unittest
class TestNotebook(unittest.TestCase):
    def testYearlyTotals(self):
        actual = dataHelper.getSurveyStatsV1('2021').tail(1)
        # compare 2021 yearly totals with expected values
        self.assertEqual(actual['running_total_all_salmon'].item(), 1008)
        self.assertEqual(actual['running_total_all_chum'].item(), 939)
        self.assertEqual(actual['running_total_all_coho'].item(), 66)
        self.assertEqual(actual['Survey_Date'].item(), '2021-12-07')
    def testSurveyStats(self):
        # compare 2021-11-16 against expected
        actual = dataHelper.getSurveyStatsV1('2021').query('`Survey_Date` == "2021-11-16"')
        self.assertEqual(actual['dead_chum_count'].item(), 114)
        self.assertEqual(actual['dead_coho_count'].item(), 29)
        self.assertEqual(actual['live_chum_count'].item(), 447)
        self.assertEqual(actual['live_coho_count'].item(), 2)
        self.assertEqual(actual['live_cutthroat_count'].item(), 2)
        self.assertEqual(actual['redd_count'].item(), 39)
        self.assertEqual(actual['total_dead_salmon_count'].item(), 143)
        self.assertEqual(actual['total_live_salmon_count'].item(), 451)
        self.assertEqual(actual['running_total_dead_salmon'].item(), 277)
        self.assertEqual(actual['running_total_dead_chum'].item(), 222)
        self.assertEqual(actual['running_total_dead_coho'].item(), 52)
if not aboveDamOnly: unittest.main(argv=[''], exit=False)

In [ ]:
if use_v2_report:
    yearByYearCountPlot = reportHelper.getInteractiveYearByYearCountPlot()
    display(HTML(yearByYearCountPlot))
else:
    yearByYearCountPlot = reportHelper.getYearByYearCountPlot()

In [ ]:
## ALL SURVEYS SCATTER MAP
if use_v2_report:
    scatterMapByYear = reportHelper.displayScatterMapByYear()
    display(HTML(scatterMapByYear))
else:
    df = dataHelper.getYearScatterMapData(surveyYear)
    yearScatterMap = reportHelper.getScatterMap(df, f'{surveyYear} Fish Scatter Map')

In [ ]:
##LATEST SURVEY SCATTER MAP
if use_v2_report:
    latestScatterMap = reportHelper.displayLatestScatterMap()
    display(HTML(latestScatterMap))
else:
    df = dataHelper.getLatestScatterMapData()
    latestSurvey = df.iloc[0]['Survey_Date']
    latestScatterMap = reportHelper.getScatterMap(df, f'{latestSurvey} Fish Scatter Map')

In [ ]:
## GENERATE REPORT WITH WHICHEVER FIGURES WERE CREATED
from jinja2 import Environment, FileSystemLoader
currentTimePacific = datetime.now(pytz.timezone('America/Los_Angeles')).strftime('%Y-%m-%d_%H-%M-%S') #cannot use system time due to colab
reportFileName = currentTimePacific + '_salmonReport.html'
def generateReport():
    if IN_COLAB:
        templatePath = 'Survey-Notebook/templates'
    else:
        templatePath = 'templates'
    env = Environment(loader=FileSystemLoader(templatePath))
    template = env.get_template(f"report_template{'_v2' if use_v2_report else ''}.html")
    reportData = {}
    reportData['reportGenTime'] = currentTimePacific
    reportData['reddsTable'] = reddsTable
    reportData['yearByYearCountPlot'] = yearByYearCountPlot
    reportData['countPlot'] = countPlot
    reportData['surveyStatsTable'] = surveyStatsTable
    reportData['yearScatterMap'] = yearScatterMap
    reportData['latestScatterMap'] = latestScatterMap
    reportData['maxSurveyChum'] = maxSurveyChum
    reportData['maxSurveyChumDate'] = maxSurveyChumDate
    reportData['maxSurveyCoho'] = maxSurveyCoho
    reportData['maxSurveyCohoDate'] = maxSurveyCohoDate
    reportData['chumSpawnSuccess'] = chumSpawnSuccess
    reportData['cohoSpawnSuccess'] = cohoSpawnSuccess
    reportData['chumFemaleSpawnSuccess'] = chumFemaleSpawnSuccess
    reportData['chumMaleSpawnSuccess'] = chumMaleSpawnSuccess
    reportData['chumPredation'] = chumPredation
    reportData['cohoPredation'] = cohoPredation
    reportData['countPlotByYear'] = countPlotByYear
    reportData['spawningChartsByYear'] = spawningChartsByYear
    reportData['predationChartsByYear'] = predationChartsByYear
    reportData['scatterMapByYear'] = scatterMapByYear
    html = template.render(reportData)
    with open(reportFileName, 'w') as f:
        f.write(html)
generateReport()

In [ ]:
#Download HTML report if in google colab
if IN_COLAB:
    from google.colab import files
    files.download(reportFileName)